In [2]:
import os
print(os.getcwd ())
print(os.listdir("."))

c:\Users\Guest User\Desktop\my-first-git-project\Mini_Project_2\Pyspark_RDD
['.venv', 'coffee_sales.csv', 'Spotify data.csv']


In [ ]:
import sys
import os
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("SpotifyAnalysis") \
    .master("local[2]") \
    .config("spark.python.worker.timeout", "120") \
    .getOrCreate()
sc = spark.sparkContext
print("Spark:", spark.version)
print("Python:", sys.executable)

Spark: 4.3.0
Python: c:\Users\Guest User\Desktop\my-first-git-project\Mini_Project_2\Pyspark_RDD\.venv\Scripts\python.exe


In [6]:
import csv
with open("Spotify data.csv", encoding="utf-8", newline="") as f:
    rows = list(csv.reader(f))
rdd = sc.parallelize(rows[1:])
songs_per_artist = (
    rdd.map(lambda row: (row[16], 1))
       .reduceByKey(lambda a, b: a + b)
)
print(songs_per_artist.collect())

[('Future', 8), ('Junior Boys', 1), ('Drake', 16), ('Honey Claws', 1), ('Lil Yachty', 1), ('Crystal Castles', 9), ('New Order', 5), ('Le Youth', 1), ('Young Thug', 7), ('Ty Segall', 4), ('Myron & E', 1), ('TR/ST', 4), ('A-Trak', 2), ('Father', 3), ('2milly', 1), ('Ty Dolla $ign', 2), ('Three 6 Mafia', 4), ('Rockwell', 1), ('Kanye West', 8), ('Young M.A.', 1), ('Mark Morrison', 1), ('Dr. John', 1), ('T. P. Orchestre Poly-Rythmo', 3), ('Ebo Taylor', 3), ('Future Islands', 3), ('Jamie xx', 4), ('Vince Staples', 2), ('Best Coast', 2), ("Ariel Pink's Haunted Graffiti", 2), ('Bobby "Blue" Bland', 1), ('Portugal. The Man', 1), ('Sweet Valley', 1), ('Ezale', 3), ('Amadou & Mariam', 1), ('The Sonics', 3), ('Thundercat', 2), ('How Sad', 1), ('Kool & The Gang', 1), ('James Brown', 3), ("Fool's Gold", 2), ('Punkin Machine', 1), ('Swishahouse', 1), ('Tame Impala', 5), ('Kwabs', 1), ('The Clash', 1), ('Tijuana Panthers', 1), ('Lady Leshurr', 1), ('Bassnectar', 3), ('Rick Ross', 13), ('Brown Bird', 1

In [9]:
danceability = rdd.map(lambda row: float(row[2]))
total_danceability = danceability.reduce(lambda a, b: a + b)
average_danceability = total_danceability / rdd.count()
print("Average Danceability:", average_danceability)

Average Danceability: 0.6184219137332679


In [15]:
love_songs_count = spotify_rdd.filter(lambda cols: "love" in cols[15].lower()).count()
print("Songs with 'love':", love_songs_count)

Songs with 'love': 88


In [17]:
for index, value in enumerate(spotify_rdd.first()):
    print(f"Index {index}: {value}")

Index 0: 0
Index 1: 0.0102
Index 2: 0.833
Index 3: 204600
Index 4: 0.434
Index 5: 0.0219
Index 6: 2
Index 7: 0.165
Index 8: -8.795
Index 9: 1
Index 10: 0.431
Index 11: 150.062
Index 12: 4
Index 13: 0.286
Index 14: 1
Index 15: Mask Off
Index 16: Future


In [18]:
def is_party_song(cols):
    try:
        # Change [1] and [3] to [2] and [4] if Step 1 showed an extra index column at 0!
        danceability = float(cols[1])
        energy = float(cols[3])
        return danceability > 0.75 and energy > 0.75
    except (ValueError, IndexError):
        return False

# Change [14], [15] to [15], [16] if Step 1 showed an extra index column at 0!
party_songs = spotify_rdd.filter(is_party_song) \
                         .map(lambda cols: (cols[14], cols[15])) \
                         .take(15)

print("5th Birthday Party Playlist:", party_songs)

5th Birthday Party Playlist: [('1', 'Sabali'), ('1', "Words I Don't Remember"), ('1', 'Demon'), ('1', 'Look at You'), ('1', 'Bang! Bang!'), ('1', "Words I Don't Remember"), ('1', 'Bam Bam'), ('1', 'Night on Bald Mountain'), ('1', 'Indian Food'), ('1', 'Childs Play (feat. Chance the Rapper)'), ('1', 'Hunger Of The Pine'), ('1', 'Baby'), ('1', 'Calgary'), ('1', 'Cloud Atlas End Title'), ('1', 'Loner')]
